# 16. Stacking Feature Ablation Study

**Tujuan:** Evaluasi Stacking Ensemble pada berbagai jumlah fitur.

**Cache:** Jika `stacking_ablation_16.pkl` sudah ada, langsung load (skip training).

In [ ]:
import sys
!{sys.executable} -m pip install lightgbm catboost scikit-learn xgboost matplotlib -q

import numpy as np
import pandas as pd
import pickle
import os
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, matthews_corrcoef, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
TEST_SIZE = 0.20
DATA_DIR = '../data/'
CACHE_FILE = os.path.join(DATA_DIR, 'stacking_ablation_16.pkl')

print('Libraries loaded.')

In [ ]:
# === CHECK CACHE ===
if os.path.exists(CACHE_FILE):
    print(f'Cache found: {CACHE_FILE}')
    print('Loading cached results (skip training)...')
    with open(CACHE_FILE, 'rb') as f:
        cached = pickle.load(f)
    ablation_results = cached['ablation_results']
    ablation_df = pd.DataFrame(ablation_results)
    ranked_indices = cached['ranked_indices']
    ranked_features = cached['ranked_features']
    SKIP_TRAINING = True
    print('Done. Jump to visualization cells.')
else:
    print('No cache found. Will run full ablation.')
    SKIP_TRAINING = False

In [ ]:
# === LOAD DATA (only if no cache) ===
if not SKIP_TRAINING:
    with open(os.path.join(DATA_DIR, 'cleaned_25.pkl'), 'rb') as f:
        data = pickle.load(f)
    X = data['X']
    y = data['y']
    feature_names = data.get('feature_names', [f'f{i}' for i in range(X.shape[1])])

    with open(os.path.join(DATA_DIR, 'stacking_baseline_15.pkl'), 'rb') as f:
        stack_data = pickle.load(f)

    xgb_model = stack_data['base_models']['XGBoost']
    importances = xgb_model.feature_importances_
    ranked_indices = np.argsort(importances)[::-1]
    ranked_features = [feature_names[i] for i in ranked_indices]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
    )
    n_classes = len(np.unique(y_train))
    print(f'Data loaded: {X.shape}, {n_classes} classes')

In [ ]:
# === RUN ABLATION (only if no cache) ===
if not SKIP_TRAINING:
    def train_stacking_simple(X_tr, X_te, y_tr, y_te, n_classes, config_name):
        base_learners = [
            ('XGB', XGBClassifier(max_depth=6, n_estimators=100, learning_rate=0.1,
                                  use_label_encoder=False, eval_metric='mlogloss',
                                  random_state=RANDOM_SEED, verbosity=0)),
            ('LGBM', LGBMClassifier(max_depth=6, n_estimators=100, learning_rate=0.1,
                                    random_state=RANDOM_SEED, verbose=-1)),
            ('CAT', CatBoostClassifier(depth=6, iterations=100, learning_rate=0.1,
                                       random_seed=RANDOM_SEED, verbose=0))
        ]
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
        meta_tr = np.zeros((X_tr.shape[0], n_classes * 3))
        meta_te = np.zeros((X_te.shape[0], n_classes * 3))
        start = time.time()
        for idx, (nm, mdl) in enumerate(base_learners):
            oof = np.zeros((X_tr.shape[0], n_classes))
            te_p = np.zeros((X_te.shape[0], n_classes))
            for ti, vi in cv.split(X_tr, y_tr):
                m = mdl.__class__(**mdl.get_params())
                m.fit(X_tr[ti], y_tr[ti])
                oof[vi] = m.predict_proba(X_tr[vi])
                te_p += m.predict_proba(X_te) / cv.n_splits
            meta_tr[:, idx*n_classes:(idx+1)*n_classes] = oof
            meta_te[:, idx*n_classes:(idx+1)*n_classes] = te_p
        scl = StandardScaler()
        lr = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, multi_class='multinomial')
        lr.fit(scl.fit_transform(meta_tr), y_tr)
        y_pred = lr.predict(scl.transform(meta_te))
        total_time = time.time() - start
        return {
            'config': config_name,
            'n_features': X_tr.shape[1],
            'f1_score': f1_score(y_te, y_pred, average='weighted'),
            'mcc': matthews_corrcoef(y_te, y_pred),
            'accuracy': accuracy_score(y_te, y_pred),
            'train_time': total_time
        }

    feature_configs = {
        'Top-15': ranked_indices[:15].tolist(),
        'Top-10': ranked_indices[:10].tolist(),
        'Top-5': ranked_indices[:5].tolist(),
    }
    ablation_results = []
    print('='*60)
    print('  STACKING FEATURE ABLATION')
    print('='*60)
    for cfg, idx_list in feature_configs.items():
        print(f'\n  {cfg} ({len(idx_list)} features)...')
        res = train_stacking_simple(X_train[:, idx_list], X_test[:, idx_list],
                                    y_train, y_test, n_classes, cfg)
        ablation_results.append(res)
        print(f'    F1={res["f1_score"]*100:.2f}% | MCC={res["mcc"]:.4f} | Time={res["train_time"]:.0f}s')
    ablation_df = pd.DataFrame(ablation_results)
    # Save cache
    with open(CACHE_FILE, 'wb') as f:
        pickle.dump({'ablation_results': ablation_results, 'ablation_df': ablation_df,
                     'ranked_indices': ranked_indices, 'ranked_features': ranked_features}, f)
    print(f'\nSaved cache: {CACHE_FILE}')

In [ ]:
# === RESULTS TABLE ===
print('\n' + '='*60)
print('  ABLATION RESULTS')
print('='*60)
print(pd.DataFrame(ablation_results)[['config','n_features','f1_score','mcc','train_time']].to_string(index=False))

In [ ]:
# === VISUALIZATION ===
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
configs = [r['config'] for r in ablation_results]
x = np.arange(len(configs))
axes[0].bar(x, [r['f1_score']*100 for r in ablation_results], color='steelblue')
axes[0].set_xticks(x); axes[0].set_xticklabels(configs)
axes[0].set_ylabel('F1 (%)'); axes[0].set_title('Stacking F1 vs Features')
axes[0].set_ylim(90, 101); axes[0].grid(axis='y', alpha=0.3)
axes[1].bar(x, [r['mcc'] for r in ablation_results], color='orange')
axes[1].set_xticks(x); axes[1].set_xticklabels(configs)
axes[1].set_ylabel('MCC'); axes[1].set_title('Stacking MCC vs Features')
axes[1].set_ylim(0.85, 1.0); axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'stacking_ablation_f1_mcc.png'), bbox_inches='tight')
plt.show()
print('Saved: stacking_ablation_f1_mcc.png')
print('\nNotebook 16 selesai.')